# PHEMS No-Label-FFill Training and Submission

This notebook trains CatBoost using only official labeled rows, keeps unlabeled rows for feature construction, and writes the final submission CSV.

Output folder: `C:/Users/pjs87/phems_outputs/run_20260425_no_label_ffill`


## Imports and paths


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupKFold

ROOT = Path(r"C:\Users\pjs87\guswls")
SOURCE = Path(r"C:\Users\pjs87\phems_outputs\run_20260425_1337")
OUT = Path(r"C:\Users\pjs87\phems_outputs\run_20260425_no_label_ffill")
NOTEBOOK = ROOT / "final-phems-solution__3_.ipynb"


## Logging and notebook feature loader


In [ ]:
def log(msg: str) -> None:
    print(msg, flush=True)


def load_add_features():
    nb = json.loads(NOTEBOOK.read_text(encoding="utf-8"))
    for cell in nb["cells"]:
        src = "".join(cell.get("source", []))
        if "def add_features" in src:
            env = {"pd": pd, "np": np}
            exec(src, env)
            return env["add_features"]
    raise RuntimeError("add_features not found")


## ID helper


In [ ]:
def person_datetime_id(df: pd.DataFrame) -> pd.Series:
    return df["person_id"].astype(int).astype(str) + "_" + pd.to_datetime(df["measurement_datetime"]).astype(str)


## Pipeline


In [ ]:
def main() -> None:
    OUT.mkdir(parents=True, exist_ok=True)
    add_features = load_add_features()

    log("[run] Loading feature tables")
    train_df = pd.read_csv(SOURCE / "train_data_w_features_FINAL.csv")
    test_df = pd.read_csv(SOURCE / "test_data_w_features_FINAL.csv")
    labels = pd.read_csv(ROOT / "data" / "training_data" / "SepsisLabel_train.csv")
    test_index = pd.read_csv(ROOT / "data" / "testing_data" / "SepsisLabel_test.csv")

    train_df["measurement_datetime"] = pd.to_datetime(train_df["measurement_datetime"])
    test_df["measurement_datetime"] = pd.to_datetime(test_df["measurement_datetime"])
    labels["measurement_datetime"] = pd.to_datetime(labels["measurement_datetime"])
    test_index["measurement_datetime"] = pd.to_datetime(test_index["measurement_datetime"])

    log("[run] Applying post-pipeline add_features")
    train_df = add_features(train_df)
    test_df = add_features(test_df)

    # Fix #2: use only official label rows. Duplicate labels are all equal except one conflict;
    # max preserves the manual correction used in the notebook for 2024-09-09 09:00.
    labels = labels.groupby(["person_id", "measurement_datetime"], as_index=False)["SepsisLabel"].max()
    train_df = train_df.drop(columns=["sepsislabel", "SepsisLabel"], errors="ignore").merge(
        labels.rename(columns={"SepsisLabel": "sepsislabel"}),
        on=["person_id", "measurement_datetime"],
        how="inner",
    )

    y = train_df["sepsislabel"].astype(int)
    drop_cols = [
        "drug_concept_id", "route_concept_id", "drug_str", "route_str",
        "sepsislabel", "person_id", "measurement_datetime", "visit_occurrence_id",
    ]
    X = train_df.drop(columns=drop_cols, errors="ignore").copy()
    categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
    for col in categorical_features:
        X[col] = X[col].fillna("missing").astype(str)

    X_test = test_df.drop(columns=["person_id_datetime"] + drop_cols, errors="ignore").copy()
    for col in X.columns:
        if col not in X_test.columns:
            X_test[col] = 0
    extra_cols = [col for col in X_test.columns if col not in X.columns]
    if extra_cols:
        X_test = X_test.drop(columns=extra_cols)
    X_test = X_test[X.columns]
    for col in categorical_features:
        X_test[col] = X_test[col].fillna("missing").astype(str)

    groups = train_df["person_id"]
    gkf = GroupKFold(n_splits=5)
    params = {
        "iterations": 1000,
        "learning_rate": 0.01,
        "depth": 6,
        "task_type": "CPU",
        "eval_metric": "PRAUC",
        "early_stopping_rounds": 100,
        "class_weights": {0: 1.0, 1: 110.0},
        "verbose": 100,
    }

    log(f"[run] Train rows={len(X)} positives={int(y.sum())} features={X.shape[1]} cats={categorical_features}")
    oof = np.zeros(len(X), dtype=float)
    test_preds = []
    folds = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), start=1):
        log(f"[run] Training fold {fold}")
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        model = CatBoostClassifier(**params)
        model.fit(
            Pool(X_train, y_train, cat_features=categorical_features),
            eval_set=Pool(X_val, y_val, cat_features=categorical_features),
            verbose=100,
        )
        val_pred = model.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_pred
        prauc = float(average_precision_score(y_val, val_pred))
        best = float(model.best_score_["validation"]["PRAUC"])
        folds.append({"fold": fold, "average_precision": prauc, "catboost_prauc": best, "n_val": int(len(val_idx)), "pos_val": int(y_val.sum())})
        log(f"[run] Fold {fold} AP={prauc} CatBoostPRAUC={best}")
        model.save_model(str(OUT / f"catboost_fold_{fold}.cbm"))
        test_preds.append(model.predict_proba(Pool(X_test, cat_features=categorical_features))[:, 1])

    mean_pred = np.mean(np.vstack(test_preds), axis=0)
    pred_df = pd.DataFrame({"person_id_datetime": person_datetime_id(test_df), "SepsisLabel": mean_pred}).drop_duplicates("person_id_datetime")
    test_index["person_id_datetime"] = person_datetime_id(test_index)
    submission = test_index[["person_id_datetime"]].merge(pred_df, on="person_id_datetime", how="left")
    missing = int(submission["SepsisLabel"].isna().sum())
    if missing:
        raise RuntimeError(f"Missing predictions: {missing}")
    submission.to_csv(OUT / "submission.csv", index=False)

    metrics = {
        "train_rows": int(len(X)),
        "positive_rows": int(y.sum()),
        "n_features": int(X.shape[1]),
        "categorical_features": categorical_features,
        "folds": folds,
        "mean_fold_average_precision": float(np.mean([f["average_precision"] for f in folds])),
        "mean_fold_catboost_prauc": float(np.mean([f["catboost_prauc"] for f in folds])),
        "oof_average_precision": float(average_precision_score(y, oof)),
        "submission_rows": int(len(submission)),
        "submission_mean": float(submission["SepsisLabel"].mean()),
    }
    (OUT / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    log("[run] DONE")
    log(json.dumps(metrics, indent=2))


## Run


In [ ]:
if __name__ == "__main__":
    main()
